# S99: Variant consistency check (no-FTS variant vs main v1.0.1)

Report only. Reads the variant feature log (`config.tables.feature_log`) and the main v1.0.1 feature log (read-only),
and writes only under the variant `artifacts_root` (`reports/99_variant_consistency.xlsx`, `reports/run_summaries/s99_variant_consistency.json`).

1. Every FTS feature has `L0_reason = SOURCE_EXCLUDED_BY_SCOPE` in the variant (asserted).
2. For every other feature the L0–L3 columns are identical in both logs, tolerance 1e-9 (asserted).
3. Features whose L4 / L5 outcome differs between the runs, with the main run's `corr_with` (report only).
4. `funnel_config_hash` equals the v1.0.1 value from the locked main registry (asserted).

In [0]:
%pip install "openpyxl" --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, time, hashlib, shutil, tempfile, importlib
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)

ARTIFACTS      = common.artifacts_path(cfg)          # variant root: the only place this notebook writes
MAIN_ARTIFACTS = common.main_artifacts_path(cfg)     # main v1.0.1 root: read-only
REPORTS        = os.path.join(ARTIFACTS, "reports")
RUN_SUMS       = os.path.join(REPORTS, "run_summaries")
XLSX_PATH      = os.path.join(REPORTS, "99_variant_consistency.xlsx")
SUMMARY_PATH   = os.path.join(RUN_SUMS, "s99_variant_consistency.json")

IS_VARIANT = common.variant_guard(cfg)               # artifacts_root / feature log differ from the main model
assert IS_VARIANT, "s99 compares a variant with the main model, but config is not a variant run " \
                   "(artifacts_root == main_artifacts_root)"
for p in (XLSX_PATH, SUMMARY_PATH):
    assert p.startswith(ARTIFACTS.rstrip("/") + "/") and not p.startswith(MAIN_ARTIFACTS.rstrip("/") + "/"), \
        f"output {p} is not under the variant artifacts_root"
os.makedirs(RUN_SUMS, exist_ok=True)
RUN_CTX = common.print_run_context(cfg, PROJECT_ROOT)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
VAR_FLOG_TBL  = common.table_name(cfg, "feature_log")
MAIN_FLOG_REQ = f"{cat}.{sch}.acard_feature_log_v101"    # main log named in the request
MAIN_FLOG_DEF = common.main_table_name(cfg, "feature_log")  # main log written by v1.0.1 (default name)
MAIN_REGISTRY_PATH = os.path.join(MAIN_ARTIFACTS, "model_registry.json")

TOL = 1e-9
STRICT_COLS = ["L0_pass", "L0_reason", "single_value_share", "missing_rate", "iv_dev", "iv_cv", "n_bins", "trend",
               "direction_conflict", "iv_s1", "iv_s2", "iv_s3", "iv_s4", "rank_corr_min", "head_tail_flip", "L2_flag",
               "L2_pass", "psi_max_dev", "psi_last_q", "max_adj_psi", "psi_fwd", "psi_status", "cause_type",
               "psi_decision", "L3_pass"]
L45_OUTCOME = ["L4_pass", "L5_pass"]                   # outcome columns compared for point 3
L45_DETAIL  = ["L4_reason", "L5_reason"]

GIT_COMMIT  = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
FUNNEL_HASH = hashlib.sha256(json.dumps(cfg["funnel"], sort_keys=True).encode()).hexdigest()
_TMP = tempfile.mkdtemp(prefix="s99_")
_t0 = time.time()
print(f"Git commit : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]} | Funnel hash: {FUNNEL_HASH[:12]}")
print(f"Variant log: {VAR_FLOG_TBL}")
print(f"Report     : {XLSX_PATH}")

In [0]:
# ── Main registry (locked v1.0.1) is the reference for the main log and the funnel hash ──
with open(MAIN_REGISTRY_PATH) as fh:
    MAIN_REG = json.load(fh)
assert MAIN_REG.get("locked") is True, f"main registry {MAIN_REGISTRY_PATH} is not locked"
print(f"Main registry: v{MAIN_REG['version']} locked at {MAIN_REG.get('locked_at')}")
vc_main = (cfg.get("variant_comparison") or {}).get("main_registry")
if vc_main and os.path.abspath(vc_main) != os.path.abspath(MAIN_REGISTRY_PATH):
    print(f"\u26a0 config.variant_comparison.main_registry = {vc_main} differs from {MAIN_REGISTRY_PATH} "
          f"(exists: {os.path.exists(vc_main)}); this check uses <main_artifacts_root>/model_registry.json")

# ── Main feature log (read-only). acard_feature_log_v101 if it exists, else the default main log, which is
#    accepted only if it reproduces the locked registry (Model A features and B/C pool). ──
MAIN_FLOG_TBL = MAIN_FLOG_REQ if spark.catalog.tableExists(MAIN_FLOG_REQ) else MAIN_FLOG_DEF
if MAIN_FLOG_TBL != MAIN_FLOG_REQ:
    print(f"\u26a0 {MAIN_FLOG_REQ} does not exist; using {MAIN_FLOG_TBL} (the log written by the v1.0.1 run)")
main_log = spark.table(MAIN_FLOG_TBL).toPandas()
var_log  = spark.table(VAR_FLOG_TBL).toPandas()
for df in (main_log, var_log):
    df.attrs = {}
    assert df["feature"].is_unique, "feature column is not unique in a feature log"

main_A    = set(main_log.loc[main_log["final_A"].astype(str).str.upper() == "Y", "feature"]) if "final_A" in main_log else set()
main_pool = set(main_log.loc[main_log["L4_pass"].astype(str).str.upper() == "Y", "feature"]) if "L4_pass" in main_log else set()
REG_A, REG_POOL = set(MAIN_REG["features"]["A"]), set(MAIN_REG["features"]["BC_pool"])
MAIN_LOG_MATCHES_REGISTRY = (main_A == REG_A) and (main_pool == REG_POOL)
print(f"Main log {MAIN_FLOG_TBL}: {len(main_log)} rows | final_A {len(main_A)} vs registry A {len(REG_A)} | "
      f"L4 pool {len(main_pool)} vs registry {len(REG_POOL)} -> matches registry: {MAIN_LOG_MATCHES_REGISTRY}")
print(f"Variant log {VAR_FLOG_TBL}: {len(var_log)} rows, {len(var_log.columns)} columns")
SAME_FEATURES = set(main_log["feature"]) == set(var_log["feature"])
print(f"Same feature rows in both logs: {SAME_FEATURES} "
      f"(only main {len(set(main_log.feature) - set(var_log.feature))}, only variant {len(set(var_log.feature) - set(main_log.feature))})")

M = main_log.set_index("feature")
V = var_log.set_index("feature")
COMMON = sorted(set(M.index) & set(V.index))

In [0]:
# FTS features: config.feature_scope.exclude_prefixes applied to feature_dict (prefix / source / description, same rule
# as E4) plus any feature-log row whose name contains the prefix stem or whose source is the stem (e.g. FTS).
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
scope = common.scope_exclusions(fdict[fdict["role"] == "FEATURE"], cfg)
prefixes = [str(p).lower() for p in ((cfg.get("feature_scope") or {}).get("exclude_prefixes") or []) if p]
assert prefixes, "config.feature_scope.exclude_prefixes is empty: no FTS scope defined for the variant"
stems = sorted({p.strip("_") for p in prefixes if p.strip("_")})

by_log = set(V.index[[any(s in f.lower() for s in stems) for f in V.index]]) | \
         set(V.index[V["source"].astype(str).str.strip().str.lower().isin(stems)])
FTS = sorted(set(scope["feature"]) | by_log)
basis = scope.set_index("feature")["basis"].to_dict()

fts_rows = pd.DataFrame({
    "feature": FTS,
    "basis": [basis.get(f, "feature_log name/source") for f in FTS],
    "in_variant_log": [f in V.index for f in FTS],
    "variant_L0_pass": [V.at[f, "L0_pass"] if f in V.index else None for f in FTS],
    "variant_L0_reason": [V.at[f, "L0_reason"] if f in V.index else None for f in FTS],
    "main_L0_reason": [M.at[f, "L0_reason"] if f in M.index else None for f in FTS],
})
fts_rows["ok"] = fts_rows["in_variant_log"] & (fts_rows["variant_L0_reason"] == common.SCOPE_EXCLUDED_REASON) \
                 & (fts_rows["variant_L0_pass"] == "N")
CHECK1_OK = bool(len(fts_rows) > 0 and fts_rows["ok"].all())
print(f"FTS features: {len(FTS)} (by basis: {fts_rows['basis'].value_counts().to_dict()})")
print(f"Check 1 — all have L0_reason = {common.SCOPE_EXCLUDED_REASON}: {CHECK1_OK} "
      f"({int((~fts_rows['ok']).sum())} not)")
if not CHECK1_OK:
    display(fts_rows[~fts_rows["ok"]])

In [0]:
_NULLS = {"", "nan", "none", "nat", "<na>", "null"}

def _norm(v):
    """Null-like values (None, NaN, 'nan', 'None', '') -> None; other values -> stripped string."""
    if v is None or (isinstance(v, float) and np.isnan(v)) or v is pd.NA:
        return None
    s = str(v).strip()
    return None if s.lower() in _NULLS else s

def _as_num(s):
    """Numeric version of a column if every non-null value is numeric, else None."""
    n = s.map(_norm)
    out = pd.to_numeric(n, errors="coerce")
    return out if (n.isna() | out.notna()).all() else None

OTHER = [f for f in COMMON if f not in set(FTS)]
miss_cols = [c for c in STRICT_COLS if c not in M.columns or c not in V.columns]
mismatch_rows, col_summary = [], []
for c in [c for c in STRICT_COLS if c not in miss_cols]:
    m, v = M.loc[OTHER, c], V.loc[OTHER, c]
    mn, vn = _as_num(m), _as_num(v)
    numeric = (mn is not None and vn is not None
               and (pd.api.types.is_numeric_dtype(m) or pd.api.types.is_numeric_dtype(v)))
    if numeric:
        diff = (mn - vn).abs()
        eq = (mn.isna() & vn.isna()) | (diff <= TOL)
    else:
        diff = pd.Series(np.nan, index=m.index)
        eq = m.map(_norm) == v.map(_norm)
        eq |= m.map(_norm).isna() & v.map(_norm).isna()
    col_summary.append({"column": c, "type": "numeric" if numeric else "text", "n_compared": len(OTHER),
                        "n_mismatch": int((~eq).sum()),
                        "max_abs_diff": float(diff.max()) if numeric and diff.notna().any() else None})
    for f in eq.index[~eq]:
        mismatch_rows.append({"feature": f, "column": c, "main": m.at[f], "variant": v.at[f],
                              "abs_diff": float(diff.at[f]) if numeric and pd.notna(diff.at[f]) else None})

col_summary = pd.DataFrame(col_summary)
mismatches = pd.DataFrame(mismatch_rows, columns=["feature", "column", "main", "variant", "abs_diff"])
CHECK2_OK = bool(SAME_FEATURES and not miss_cols and len(mismatches) == 0)
print(f"Non-FTS features compared: {len(OTHER)} on {len(STRICT_COLS) - len(miss_cols)} columns (tolerance {TOL})")
if miss_cols:
    print(f"\u274c Columns missing from a log: {miss_cols}")
print(f"Check 2 — identical: {CHECK2_OK} ({len(mismatches)} mismatching cells, "
      f"{mismatches['feature'].nunique() if len(mismatches) else 0} features)")
display(col_summary)
if len(mismatches):
    display(mismatches.head(50))

In [0]:
VAR_L45_RUN = all(c in V.columns for c in L45_OUTCOME)
L45_COLS = ["feature", "is_fts"] + [f"{s}_{c}" for c in L45_OUTCOME + L45_DETAIL for s in ("main", "variant")] \
           + ["main_corr_with", "main_corr_r"]
if VAR_L45_RUN:
    rows = []
    for f in COMMON:
        mo = [_norm(M.at[f, c]) if c in M.columns else None for c in L45_OUTCOME]
        vo = [_norm(V.at[f, c]) for c in L45_OUTCOME]
        if mo != vo:
            r = {"feature": f, "is_fts": f in set(FTS)}
            for c in L45_OUTCOME + L45_DETAIL:
                r[f"main_{c}"] = _norm(M.at[f, c]) if c in M.columns else None
                r[f"variant_{c}"] = _norm(V.at[f, c]) if c in V.columns else None
            r["main_corr_with"] = _norm(M.at[f, "corr_with"]) if "corr_with" in M.columns else None
            r["main_corr_r"] = M.at[f, "corr_r"] if "corr_r" in M.columns else None
            rows.append(r)
    l45_diff = pd.DataFrame(rows, columns=L45_COLS)
    L45_NOTE = f"{len(l45_diff)} feature(s) with a different L4/L5 outcome ({int(l45_diff['is_fts'].sum()) if len(l45_diff) else 0} FTS)"
else:
    l45_diff = pd.DataFrame(columns=L45_COLS)
    L45_NOTE = (f"Variant L4/L5 not run yet: {[c for c in L45_OUTCOME if c not in V.columns]} missing from "
                f"{VAR_FLOG_TBL}. Re-run s99 after s03_L4 and s03_L5 on the variant.")
print(f"Check 3 (report only) — {L45_NOTE}")
if len(l45_diff):
    display(l45_diff)

In [0]:
V101_FUNNEL_HASH = MAIN_REG["funnel_config_hash"]
hash_rows = [{"source": "main registry v" + str(MAIN_REG["version"]), "funnel_config_hash": V101_FUNNEL_HASH},
             {"source": "current config.funnel (variant)", "funnel_config_hash": FUNNEL_HASH}]
for fn in sorted(os.listdir(RUN_SUMS)):
    if fn.startswith("s03_") and fn.endswith(".json"):
        with open(os.path.join(RUN_SUMS, fn)) as fh:
            h = json.load(fh).get("funnel_config_hash")
        if h:
            hash_rows.append({"source": f"variant run summary {fn}", "funnel_config_hash": h})
hash_df = pd.DataFrame(hash_rows)
hash_df["equals_v101"] = hash_df["funnel_config_hash"] == V101_FUNNEL_HASH
CHECK4_OK = bool(hash_df["equals_v101"].all() and len(hash_df) >= 3)   # registry + config + at least E4 summary
print(f"Check 4 — funnel_config_hash equals v1.0.1 ({V101_FUNNEL_HASH[:12]}): {CHECK4_OK}")
display(hash_df)

In [0]:
summary_df = pd.DataFrame([
    {"check": "1. FTS features have L0_reason = SOURCE_EXCLUDED_BY_SCOPE", "asserted": True, "pass": CHECK1_OK,
     "detail": f"{len(FTS)} FTS features, {int((~fts_rows['ok']).sum())} not excluded"},
    {"check": "2. L0-L3 columns identical for non-FTS features (tol 1e-9)", "asserted": True, "pass": CHECK2_OK,
     "detail": f"{len(OTHER)} features x {len(STRICT_COLS) - len(miss_cols)} columns, {len(mismatches)} mismatches"
               + (f", missing columns {miss_cols}" if miss_cols else "") + ("" if SAME_FEATURES else ", feature rows differ")},
    {"check": "3. L4/L5 outcome differences (with main corr_with)", "asserted": False, "pass": None, "detail": L45_NOTE},
    {"check": "4. funnel_config_hash equals v1.0.1", "asserted": True, "pass": CHECK4_OK,
     "detail": f"v1.0.1 {V101_FUNNEL_HASH[:12]}; {int((~hash_df['equals_v101']).sum())} source(s) differ"},
])
meta = pd.DataFrame([{
    "module": "S99_variant_consistency", "run_time": datetime.now().isoformat(timespec="seconds"),
    "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH, "branch": RUN_CTX["branch"],
    "variant_artifacts_root": ARTIFACTS, "main_artifacts_root": MAIN_ARTIFACTS,
    "variant_feature_log": VAR_FLOG_TBL, "main_feature_log": MAIN_FLOG_TBL,
    "main_log_matches_registry": MAIN_LOG_MATCHES_REGISTRY, "main_registry_version": MAIN_REG["version"],
    "variant_registry_version": RUN_CTX["registry_version"], "tolerance": TOL,
}])

tmp_xl = os.path.join(_TMP, "99_variant_consistency.xlsx")
with pd.ExcelWriter(tmp_xl, engine="openpyxl") as w:
    meta.to_excel(w, sheet_name="meta", index=False)
    summary_df.to_excel(w, sheet_name="summary", index=False)
    fts_rows.to_excel(w, sheet_name="1_fts_exclusion", index=False)
    col_summary.to_excel(w, sheet_name="2_column_summary", index=False)
    mismatches.to_excel(w, sheet_name="2_mismatches", index=False)
    (l45_diff if len(l45_diff) else pd.DataFrame({"note": [L45_NOTE]})).to_excel(w, sheet_name="3_l4_l5_differences", index=False)
    hash_df.to_excel(w, sheet_name="4_funnel_hash", index=False)
shutil.copyfile(tmp_xl, XLSX_PATH)
print(f"Saved report: {XLSX_PATH}")
display(summary_df)

In [0]:
# ── Acceptance (asserts 1, 2 and 4; point 3 is report only) ──
run_summary = {
    "module": "S99_variant_consistency", "run_time": datetime.now().isoformat(timespec="seconds"),
    "elapsed_s": round(time.time() - _t0, 1), "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH,
    "funnel_config_hash": FUNNEL_HASH, "v101_funnel_config_hash": V101_FUNNEL_HASH,
    "main_feature_log": MAIN_FLOG_TBL, "variant_feature_log": VAR_FLOG_TBL,
    "main_log_matches_registry": MAIN_LOG_MATCHES_REGISTRY,
    "n_features": len(COMMON), "n_fts": len(FTS), "n_other": len(OTHER),
    "check1_fts_scope_excluded": CHECK1_OK, "check2_l0_l3_identical": CHECK2_OK,
    "check2_mismatches": len(mismatches), "check3_note": L45_NOTE, "check3_n_differences": len(l45_diff),
    "check4_funnel_hash": CHECK4_OK, "report": XLSX_PATH,
    "pass_fail": "PASS" if (CHECK1_OK and CHECK2_OK and CHECK4_OK and MAIN_LOG_MATCHES_REGISTRY) else "FAIL",
    "run_context": RUN_CTX,
}
_tmp_js = os.path.join(_TMP, "s99.json")
with open(_tmp_js, "w") as fh:
    json.dump(run_summary, fh, indent=2, default=str)
shutil.copyfile(_tmp_js, SUMMARY_PATH)
print(f"Run summary: {SUMMARY_PATH}")

assert MAIN_LOG_MATCHES_REGISTRY, (f"{MAIN_FLOG_TBL} does not reproduce the locked v{MAIN_REG['version']} registry "
                                   f"(Model A / B-C pool); it cannot be used as the main log")
assert CHECK1_OK, f"Check 1 failed: {int((~fts_rows['ok']).sum())} FTS feature(s) without L0_reason = {common.SCOPE_EXCLUDED_REASON}"
assert CHECK2_OK, (f"Check 2 failed: {len(mismatches)} mismatching cell(s) in L0-L3 columns for non-FTS features"
                   + (f"; missing columns {miss_cols}" if miss_cols else "") + ("" if SAME_FEATURES else "; feature rows differ"))
assert CHECK4_OK, f"Check 4 failed: funnel_config_hash differs from v1.0.1 ({V101_FUNNEL_HASH[:12]})"
print("\u2705 S99 checks 1, 2 and 4 passed.", L45_NOTE)